In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [2]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/NGC-3217_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/NGC-3217_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     367   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        48   (2094, 964)   float32   


In [3]:
image_cut = image[67:167, 0:1890]
image_error_cut = image_error[67:167, 0:1890]

In [4]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.03191
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [5]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [6]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/PSF/Real_seeing_NGC3217_1.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [7]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    purple_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choc_area = single_integral(x, params5, a, b)
    sum_area = blue_area + green_area +purple_area + orange_area+ choc_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (purple_area*100)/sum_area, (orange_area*100)/sum_area, (choc_area*100)/sum_area ]    

## Halpha

In [8]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))

sky_err = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Propag_errori/Sky_errors.txt")
real_errors_HA = np.sqrt(radial_profile_error_HA**2+(sky_err[2])**2+(0.04*radial_profile_HA)**2)

fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/Halpha_fit.csv", index_col=0)

In [9]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -1, fit_HA['Component 5'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

4.309390907426032
4.37492071689611
4.61567680529496
4.6731540377003675
3.6630232877831364
3.6445481848385572
1.054078514774793
1.0545506472679824
1.9435523974557467
2.062276239280878
                 Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue       9.932565e+01   0.914829   0.613479   0.279584  2.157022e-02
Green      2.381423e-14  53.752007  10.524041   0.000001  1.632234e-17
Purple     6.424884e-01  45.006882  88.230432  59.632263  3.569875e+00
Orange     0.000000e+00   0.000000   0.000000  28.002850  0.000000e+00
Chocolate  3.186629e-02   0.326281   0.632048  12.085302  9.640855e+01


In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

12.353176571675728
12.503251475296528
13.14937524119569
13.27670106045258
7.078308830679227
7.037470119790555
2.0133938618724776
2.0162969141006672
5.4558883883868265
5.734241447341257
                 Peak 1     Peak 2     Peak 3        Peak 4         Peak 5
Blue       9.928493e+01   0.974467   0.573283  2.655408e-01   2.326557e-02
Green      9.797868e-14  50.994395   6.475168  6.054098e-07   6.788210e-17
Purple     6.816182e-01  47.685630  92.246094  5.632063e+01   3.863501e+00
Orange     0.000000e+00   0.000000   0.000000  2.936314e+01  1.068149e-310
Chocolate  3.345221e-02   0.345508   0.705455  1.405069e+01   9.611323e+01


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

15.516698738592343
15.647656539053921
19.589203929661927
19.68945712062193
13.972022209548854
13.936495263208286
3.974550815417098
3.9740149704533123
6.695296303202925
6.955560087474942
                 Peak 1     Peak 2         Peak 3     Peak 4        Peak 5
Blue       9.931294e+01   1.124196   5.937913e-01   0.274442  2.813119e-02
Green      6.872583e-14  44.366293   1.045125e+01   0.000003  7.913251e-16
Purple     6.532548e-01  54.117271   8.823091e+01  58.246701  4.777029e+00
Orange     0.000000e+00   0.000000  3.863072e-223  26.859997  6.305750e-56
Chocolate  3.380609e-02   0.392240   7.240433e-01  14.618858  9.519484e+01


## HBeta

In [12]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
real_errors_HB = np.sqrt(radial_profile_error_HB**2+(sky_err[2])**2+(0.04*radial_profile_HB)**2)
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/Hbeta_fit.csv", index_col = 0)

In [13]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -1, fit_HA['Component 5'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.9759566233815116
1.9856223535163127
1.653998629499595
1.6292847333854281
2.2563878540970355
2.2976859175145035
0.875805978761715
0.876956641366498
0.5383177378349193
0.5364491366303792
                 Peak 1     Peak 2     Peak 3        Peak 4     Peak 5
Blue       9.828619e+01   0.677610   0.203693  2.620613e-02   0.002065
Green      1.688811e+00  97.122782  80.659334  1.802784e+01   6.551545
Purple     0.000000e+00   1.568123  17.268211  1.843897e-21   0.000000
Orange     1.363128e-17   0.140031   1.300938  7.448010e+01   0.860736
Chocolate  2.500200e-02   0.491454   0.567824  7.465856e+00  92.585654


In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

5.752838967114454
5.781981921341731
5.060745775867389
5.008181589579407
4.0146532690078995
4.062521129417288
1.666917198602548
1.6695542427976533
1.6172526405424952
1.6151594696337195
                 Peak 1     Peak 2     Peak 3        Peak 4     Peak 5
Blue       9.822519e+01   0.678767   0.198014  2.377530e-02   0.002110
Green      1.748930e+00  96.112790  76.254361  1.740981e+01   6.575028
Purple     0.000000e+00   2.554263  20.750122  6.458614e-22   0.000000
Orange     6.193643e-17   0.169431   2.107360  7.398857e+01   1.018536
Chocolate  2.588069e-02   0.484749   0.690143  8.577844e+00  92.404327


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


6.984888705699599
7.0135889727812195
8.579784461618106
8.54777611178132
8.145259787202889
8.248001169125155
3.2732159540753747
3.2780020105281165
1.9982627268617144
1.9947754900053236
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.816438e+01   0.706409   0.203884   0.025321   0.002722
Green      1.808877e+00  93.985939  77.679961  18.046711   7.630789
Purple     0.000000e+00   4.567547  18.964301   0.000064   0.000000
Orange     4.455003e-17   0.256027   2.462874  73.039822   1.992919
Chocolate  2.674047e-02   0.484078   0.688980   8.888082  90.373570


## NII

In [16]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/NII_fit.csv", index_col = 0)

In [17]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -1, fit_HA['Component 5'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.4599521447195913
1.4720073427287574
2.7961564676078066
2.896531495588619
2.01831697972648
2.021376757544244
0.8050722854022385
0.8055003680382467
0.7718299377340302
0.7861773271105051
                  Peak 1        Peak 2     Peak 3     Peak 4        Peak 5
Blue        9.646123e+01  6.217318e-01   0.436756   0.114381  1.253395e-02
Green       3.378235e+00  8.861338e+01  60.490140  18.307462  4.451348e+00
Purple     1.003303e-148  1.003953e+01  37.617929   0.079414  2.312158e-45
Orange      0.000000e+00  1.889734e-08   0.002757  66.900304  1.630644e-04
Chocolate   1.605344e-01  7.253591e-01   1.452418  14.598439  9.553596e+01


In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

4.231741029802779
4.259291803253538
7.972747924032909
8.209913525972519
3.7503132043765315
3.7557234865990665
1.5788724034309312
1.5801722628938084
2.2182654454043953
2.251090262571037
                  Peak 1        Peak 2     Peak 3     Peak 4        Peak 5
Blue        9.631775e+01  6.624861e-01   0.420726   0.104428  1.324779e-02
Green       3.515662e+00  8.764456e+01  56.207585  17.229529  4.668912e+00
Purple     2.711922e-125  1.092667e+01  41.689951   0.031892  2.351555e-36
Orange     3.238948e-289  5.888697e-07   0.015436  66.498542  1.068194e-03
Chocolate   1.665905e-01  7.662834e-01   1.666302  16.135608  9.531677e+01


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

5.38335207841373
5.406426422040477
11.962708128458576
12.185740607227434
7.685656457828587
7.701169782796944
3.100780985192392
3.103767849488297
2.811174092874283
2.840390964149636
                  Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue        9.637205e+01   0.761216   0.421164   0.109066  1.583101e-02
Green       3.461235e+00  85.256210  58.481684  17.819979  5.278122e+00
Purple     1.705429e-125  13.122230  39.391578   0.165766  1.760477e-28
Orange     2.036856e-289   0.000018   0.064627  65.215749  1.297684e-02
Chocolate   1.667142e-01   0.860326   1.640946  16.689439  9.469307e+01


## SII

In [20]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/SII_fit.csv", index_col = 0)

In [21]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -1, fit_HA['Component 5'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

1.1297784533537598
1.1139584214815588
1.8298882954489866
1.8620825826344691
1.371202137220257
1.3732015442495673
0.6537125661935477
0.654062446029041
0.5114687859119811
0.5149537731914976
                  Peak 1        Peak 2     Peak 3     Peak 4        Peak 5
Blue        9.649915e+01  1.207319e+00   0.959409   0.380194  9.851585e-02
Green       3.462298e+00  8.943633e+01  66.234100  18.631095  5.301875e+00
Purple     4.260607e-179  8.790687e+00  31.564595   0.046558  3.174981e-52
Orange     5.013007e-207  5.996751e-07   0.011273  67.209179  1.232366e-03
Chocolate   3.854766e-02  5.656672e-01   1.230623  13.732973  9.459838e+01


In [22]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

3.263988087180827
3.221708040436254
5.254084455485242
5.329438646573807
2.546900449295283
2.5506205690752086
1.2826997091538428
1.2837237151672396
1.4887579507952373
1.4969763597735293
                  Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue        9.634368e+01   1.271371   0.949645   0.357681  1.021690e-01
Green       3.616042e+00  88.637117  62.626195  17.514257  5.492983e+00
Purple     8.364942e-150   9.496471  34.937407   0.018375  2.257520e-41
Orange     2.036940e-185   0.000008   0.046898  66.836052  5.114998e-03
Chocolate   4.027488e-02   0.595033   1.439856  15.273636  9.439973e+01


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

4.17182910373653
4.1257695493284325
7.997898023200863
8.064372042139912
5.202041829286081
5.2096384577833295
2.5214126678274296
2.5237279836393576
1.898760828031481
1.9055784331347525
                  Peak 1     Peak 2     Peak 3     Peak 4        Peak 5
Blue        9.642678e+01   1.421575   0.943941   0.368854  1.163822e-01
Green       3.534073e+00  86.681199  64.318780  18.093133  6.194790e+00
Purple     5.235703e-150  11.233971  33.158792   0.103433  4.686148e-32
Orange     1.274941e-185   0.000122   0.149183  65.653592  4.237300e-02
Chocolate   3.914701e-02   0.663132   1.429303  15.780988  9.364645e+01


## OIII

In [24]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/NGC3217/NGC3217_1/Fit/OIII_fit.csv", index_col = 0)

In [25]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -1, fit_HA['Component 1'].iloc[3] + 1)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -1, fit_HA['Component 2'].iloc[3] + 1)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -1, fit_HA['Component 3'].iloc[3] + 1)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -1, fit_HA['Component 4'].iloc[3] + 1)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -1, fit_HA['Component 5'].iloc[3] + 1)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)


df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

2.029437839425752
2.0527231263191568
1.5156418322613225
1.482096910690486
1.8215389891769098
1.8248082152250182
1.00553561297626
1.0070288741391318
0.44002026890600554
0.4392980940268759
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.774107e+01   1.999320   0.972655   0.332048   0.163080
Green      2.253949e+00  97.065844  88.274894  16.813511  10.756873
Purple     0.000000e+00   0.162101   7.863785   0.000000   0.000000
Orange     1.315063e-10   0.431243   2.381569  76.719311   3.420908
Chocolate  4.980289e-03   0.341491   0.507098   6.135131  85.659139


In [26]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

5.837235097810461
5.909563997489498
4.611033422861297
4.5105332292591775
3.1725338907788805
3.1817881581129344
1.9153140369828152
1.918496168131168
1.32849342994899
1.3259007968620287
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.763523e+01   1.989008   1.024682   0.322341   0.162971
Green      2.359524e+00  96.700274  84.694050  16.408932  10.727713
Purple     0.000000e+00   0.496062   9.997971   0.000000   0.000000
Orange     2.236320e-10   0.474746   3.641224  76.186854   3.708735
Chocolate  5.246580e-03   0.339911   0.642074   7.081872  85.400582


In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
    fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)
data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'],
                      fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], 
                      fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), 
                        np.array(data3).reshape(-1, 1), np.array(data4).reshape(-1, 1), np.array(data5).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(5)],
                  index=['Blue', 'Green', 'Purple', 'Orange', 'Chocolate'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

6.984478271229823
7.061468396171017
7.7174319973871315
7.5649487278540075
6.5719007981949895
6.555195334717393
3.7476518495130535
3.753528744589764
1.7035775130344848
1.7007657943996777
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5
Blue       9.749521e+01   2.028247   1.005449   0.333853   0.183797
Green      2.499437e+00  95.723677  85.736372  17.006277  11.846178
Purple     0.000000e+00   1.294181   8.683242   0.000000   0.000000
Orange     1.814341e-10   0.607033   3.943676  75.298043   5.755956
Chocolate  5.351862e-03   0.346863   0.631261   7.361827  82.214069
